In [ ]:
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from const import API_key
import plotly.express as px
import os

In [ ]:
url = "http://api.openweathermap.org/data/2.5/weather"
params = {
    "appid": API_key,
    "units": "metric"  
}

In [ ]:
iran_province_capitals = {
    "Alborz": "Karaj",
    "Ardebil": "Ardabil",
    "Bushehr": "Bushehr",
    "ChaharMahallandBakhtiari": "Shahr-e Kord",
    "EastAzarbaijan": "Tabriz",
    "Esfahan": "Esfahan",
    "Fars": "Shiraz",
    "Gilan": "Rasht",
    "Golestan": "Gorgan",
    "Hamadan": "Hamadan",
    "Hormozgan": "Bandar Abbas",
    "Ilam": "Ilam ",
    "Isfahan": "Isfahan",
    "Kerman": "Kerman",                       
    "Kermanshah": "Kermanshah",
    "NorthKhorasan": "Bojnurd",
    "RazaviKhorasan": "Mashhad",
    "SouthKhorasan": "Birjand",
    "Khuzestan": "Ahvaz",
    "KohgiluyehandBuyerAhmad": "Yasuj",
    "Kordestan": "Sanandaj",
    "Lorestan": "Khorramabad",
    "Markazi": "Arak",
    "Mazandaran": "Sari",
    "Qazvin": "Qazvin",
    "Qom": "Qom",
    "Semnan": "Semnan",
    "SistanandBaluchestan": "Zahedan",
    "Tehran": "Tehran",
    "WestAzarbaijan": "Urmia",
    "Yazd": "Yazd",
    "Zanjan": "Zanjan"
}


In [ ]:
weather_data = pd.DataFrame(columns=['province', 'temp', 'humidity', 'description'])

for province, city in iran_province_capitals.items():
    params['q'] = city
    response = requests.get(url, params=params)
    data = response.json()
    if response.status_code == 200:
        new_row = pd.DataFrame({
            'Longitude ': [data['coord']['lon']],
            'Latitude ':[data['coord']['lat']],
            'province': [province],
            'temp': [data['main']['temp']],
            'humidity' : [data['main']['humidity']],
            'description' : [data['weather'][0]['description']],
            'time': pd.to_datetime(data['dt'], unit='s').floor('H')
        })
        weather_data = pd.concat([weather_data, new_row])
        
    else:
        print(f"Failed to fetch data for {city}")



In [ ]:
weather_data = weather_data.rename(columns={'province': 'NAME_1'})

In [ ]:
# Load the GeoJSON file
gdf = gpd.read_file('geojson/gadm41_IRN_1.json')

data = pd.merge(gdf, weather_data, on='NAME_1')
data.head(10)

In [ ]:
# Plot the map
fig = px.choropleth(data,
                    geojson=data.geometry,
                    locations=data.index,
                    color=data.temp, 
                    hover_name=data['NAME_1'],  
                    title="Iran and Its Provinces",
                    hover_data=['humidity'])

fig.update_geos(fitbounds="locations", visible=True)

fig.show()

In [ ]:
data.columns

In [ ]:
file_path = "weather_data.csv"

if not os.path.exists(file_path):
    # If it doesn't exist, create an empty DataFrame and save it
    df = pd.DataFrame(columns=['GID_1', 'GID_0', 'COUNTRY', 'NAME_1', 'VARNAME_1', 'NL_NAME_1',
       'TYPE_1', 'ENGTYPE_1', 'CC_1', 'HASC_1', 'ISO_1', 'geometry', 'temp',
       'humidity', 'description', 'Longitude ', 'Latitude ', 'time'])  # Empty DataFrame
    df.to_csv(file_path, index=False)  # Save to CSV

# Read the file
df = pd.read_csv("weather_data.csv")

print(df)


In [ ]:
province_data = pd.read_csv('weather_data.csv')

In [ ]:
pd.concat([province_data, data]).to_csv("weather_data.csv", index=False)